In [5]:
from pathlib import Path
from dataclasses import dataclass, asdict

@dataclass
class CFG:
    train_path: Path = Path("../data/train.csv")
    test_path: Path = Path("../data/test.csv")
    sub_path: Path = Path("../data/sample_submission.csv")

    num_fold: int = 5
    dev_mode: bool = False

    # Model parameters
    n_iter: int = 10000
    max_depth: int = -1
    num_leaves: int = 1024
    colsample_bytree: float = 0.7
    learning_rate: float = 0.02

    objective: str = 'l2'
    metric: str = 'rmse'
    verbosity: int = -1
    max_bin: int = 1024
    
    random_state: int = 42
    shuffle: bool = True
    encoded_columns_start: int = -91
    log_eval: int = 100
    early_stopping: int = 200
    
cfg = CFG() 
asdict(cfg)

{'train_path': PosixPath('../data/train.csv'),
 'test_path': PosixPath('../data/test.csv'),
 'sub_path': PosixPath('../data/sample_submission.csv'),
 'num_fold': 5,
 'dev_mode': False,
 'n_iter': 10000,
 'max_depth': -1,
 'num_leaves': 1024,
 'colsample_bytree': 0.7,
 'learning_rate': 0.02,
 'objective': 'l2',
 'metric': 'rmse',
 'verbosity': -1,
 'max_bin': 1024,
 'random_state': 42,
 'shuffle': True,
 'encoded_columns_start': -91,
 'log_eval': 100,
 'early_stopping': 200}

In [6]:
from IPython.display import display
import numpy as np
import pandas as pd
import warnings
warnings.filterwarnings('ignore')
warnings.simplefilter('ignore')

from sklearn.metrics import mean_squared_error

def calc_rmse(actual, predicted):
    return np.sqrt(mean_squared_error(actual, predicted))

re_dict = {}
re_dict['podc_dict'] = {
    'Mystery Matters': 0, 'Joke Junction': 1, 'Study Sessions': 2, 'Digital Digest': 3, 
    'Mind & Body': 4, 'Fitness First': 5, 'Criminal Minds': 6, 'News Roundup': 7, 
    'Daily Digest': 8, 'Music Matters': 9, 'Sports Central': 10, 'Melody Mix': 11, 
    'Game Day': 12, 'Gadget Geek': 13, 'Global News': 14, 'Tech Talks': 15, 
    'Sport Spot': 16, 'Funny Folks': 17, 'Sports Weekly': 18, 'Business Briefs': 19, 
    'Tech Trends': 20, 'Innovators': 21, 'Health Hour': 22, 'Comedy Corner': 23, 
    'Sound Waves': 24, 'Brain Boost': 25, "Athlete's Arena": 26, 'Wellness Wave': 27, 
    'Style Guide': 28, 'World Watch': 29, 'Humor Hub': 30, 'Money Matters': 31, 
    'Healthy Living': 32, 'Home & Living': 33, 'Educational Nuggets': 34, 
    'Market Masters': 35, 'Learning Lab': 36, 'Lifestyle Lounge': 37, 
    'Crime Chronicles': 38, 'Detective Diaries': 39, 'Life Lessons': 40, 
    'Current Affairs': 41, 'Finance Focus': 42, 'Laugh Line': 43, 
    'True Crime Stories': 44, 'Business Insights': 45, 'Fashion Forward': 46, 'Tune Time': 47
}
re_dict['genr_dict'] = {'True Crime': 0, 'Comedy': 1, 'Education': 2, 'Technology': 3, 'Health': 4, 'News': 5, 'Music': 6, 'Sports': 7, 'Business': 8, 'Lifestyle': 9}
re_dict['week_dict'] = {'Monday': 0, 'Tuesday': 1, 'Wednesday': 2, 'Thursday': 3, 'Friday': 4, 'Saturday': 5, 'Sunday': 6}
re_dict['time_dict'] = {'Morning': 10, 'Afternoon': 14, 'Evening': 17, 'Night': 21}
re_dict['sent_dict'] = {'Negative': 0, 'Neutral': 1, 'Positive': 2}


def preprocess_df(df):
    df['Episode_Num'] = df['Episode_Title'].str[8:].astype(int)  # Convert to int before log transform
    df = df.drop(columns=['Episode_Title'])

    # Convert categorical variables
    df['Genre'] = df['Genre'].replace(re_dict["genr_dict"])
    df['Podcast_Name'] = df['Podcast_Name'].replace(re_dict["podc_dict"])
    df['Publication_Day'] = df['Publication_Day'].replace(re_dict["week_dict"])
    df['Publication_Time'] = df['Publication_Time'].replace(re_dict["time_dict"])
    df['Episode_Sentiment'] = df['Episode_Sentiment'].replace(re_dict["sent_dict"])

    df.loc[df['Episode_Length_minutes']>121.0, 'Episode_Length_minutes'] = 121.0

    df['Host_Guest_Diff'] = df['Host_Popularity_percentage'] - df['Guest_Popularity_percentage']
    df['Host_Guest_Ratio'] = (df['Host_Popularity_percentage'] / df['Guest_Popularity_percentage']).replace([float('inf'), -float('inf')], pd.NA)

    if "Listening_Time_minutes" in df.columns:
        df['Listening_Episode_Diff'] = df['Episode_Length_minutes'] - df['Listening_Time_minutes']
        df['Listening_Episode_Ratio'] = (df['Episode_Length_minutes'] / df['Listening_Time_minutes']).replace([float('inf'), -float('inf')], pd.NA)

    return df


df_train = pd.read_csv(cfg.train_path, index_col='id')
df_test = pd.read_csv(cfg.test_path, index_col='id')
df_sub = pd.read_csv(cfg.sub_path, index_col='id')

df_train = preprocess_df(df_train)
df_test = preprocess_df(df_test)

# target_col = "Listening_Time_minutes"
# y_train = df_train[target_col].copy()
# df_train = df_train.drop(columns=[target_col])

# df_desc = df_train.describe()

# def feature_eng(df, df_desc=df_desc):
#     for col in df_desc.columns:
#         if df_desc[col]['std'] > 0:
#             df[col + '_log'] = df[col].apply(lambda x: np.log1p(x) if x > 0 else 0)
#             df[col + '_sqrt'] = df[col].apply(lambda x: np.sqrt(x) if x > 0 else 0)
#             df[col + '_exp'] = df[col].apply(lambda x: np.exp(x) if x > 0 else 0)

#     return df

# df_train = feature_eng(df_train)
# df_test = feature_eng(df_test)

display(df_train)
display(df_train.describe())

,Podcast_Name,Episode_Length_minutes,Genre,Host_Popularity_percentage,Publication_Day,Publication_Time,Guest_Popularity_percentage,Number_of_Ads,Episode_Sentiment,Listening_Time_minutes,Episode_Num,Host_Guest_Diff,Host_Guest_Ratio,Listening_Episode_Diff,Listening_Episode_Ratio
id,,,,,,,,,,,,,,,
0,0,NaN,0,74.81,3,21,NaN,0.0,2,31.41998,98,NaN,NaN,NaN,NaN
1,1,119.80,1,66.95,5,14,75.95,2.0,0,88.01241,26,-9.00,0.881501,31.78759,1.361172
2,2,73.90,2,69.97,1,17,8.97,0.0,0,44.92531,16,61.00,7.800446,28.97469,1.644952
3,3,67.17,3,57.22,0,10,78.70,2.0,2,46.27824,45,-21.48,0.727065,20.89176,1.451438
4,4,110.51,4,80.07,0,14,58.68,3.0,1,75.61031,86,21.39,1.364519,34.89969,1.461573
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
749995,36,75.66,2,69.36,5,10,NaN,0.0,0,56.87058,25,NaN,NaN,18.78942,1.330389
749996,19,75.75,8,35.21,5,21,NaN,2.0,1,45.46242,21,NaN,NaN,30.28758,1.666211
749997,37,30.98,9,78.58,3,10,84.89,0.0,0,15.26000,51,-6.31,0.925669,15.72000,2.030144


,Podcast_Name,Episode_Length_minutes,Genre,Host_Popularity_percentage,Publication_Day,Publication_Time,Guest_Popularity_percentage,Number_of_Ads,Episode_Sentiment,Listening_Time_minutes,Episode_Num,Host_Guest_Diff,Listening_Episode_Diff
count,750000.000000,662907.000000,750000.000000,750000.000000,750000.000000,750000.000000,603970.000000,749999.000000,750000.000000,750000.000000,750000.000000,603970.000000,662907.000000
mean,23.540307,64.504430,4.556036,59.859901,3.030805,15.671500,52.236449,1.348855,0.997969,45.437406,51.445811,7.456403,18.766443
std,13.917884,32.968121,2.965912,22.873098,2.024196,4.026379,28.451241,1.151130,0.815440,27.138306,28.085623,36.090841,13.494582
min,0.000000,0.000000,0.000000,1.300000,0.000000,10.000000,0.000000,0.000000,0.000000,0.000000,1.000000,-80.170000,-115.540000
25%,12.000000,35.730000,2.000000,39.410000,1.000000,14.000000,28.380000,0.000000,0.000000,23.178350,28.000000,-18.390000,8.273530
50%,23.000000,63.840000,5.000000,60.050000,3.000000,17.000000,53.580000,1.000000,1.000000,43.379460,52.000000,6.390000,15.713050
75%,36.000000,94.070000,7.000000,79.530000,5.000000,21.000000,76.600000,2.000000,2.000000,64.811580,75.000000,32.730000,26.710000
max,47.000000,121.000000,9.000000,119.460000,6.000000,21.000000,119.910000,103.910000,2.000000,119.970000,100.000000,113.550000,103.220440


In [57]:
df_over_ads = df_train[df_train["Number_of_Ads"] > 3]
df_over_ads["Pred_Listening_Time_minutes"] = df_over_ads["Number_of_Ads"] * 0.993
print("RMSE score:", calc_rmse(df_over_ads["Listening_Time_minutes"], df_over_ads["Number_of_Ads"]))


RMSE score: 2.313134147390121


In [9]:
df_train[df_train["Number_of_Ads"]!=0.0][df_train["Number_of_Ads"]!=1.0][df_train["Number_of_Ads"]!=2.0][df_train["Number_of_Ads"]!=3.0]

,Podcast_Name,Episode_Length_minutes,Genre,Host_Popularity_percentage,Publication_Day,Publication_Time,Guest_Popularity_percentage,Number_of_Ads,Episode_Sentiment,Listening_Time_minutes,Episode_Num,Host_Guest_Diff,Host_Guest_Ratio,Listening_Episode_Diff,Listening_Episode_Ratio
id,,,,,,,,,,,,,,,
211159,27,64.83,4,48.46,6,17,NaN,53.37,2,50.44892,83,NaN,NaN,14.38108,1.285062
247170,12,35.66,7,27.35,4,17,49.87,NaN,0,23.94516,33,-22.52,0.548426,11.71484,1.489236
283606,22,109.93,4,67.81,6,10,77.90,103.91,1,103.89696,15,-10.09,0.870475,6.03304,1.058068
436577,10,115.25,7,28.58,5,14,23.65,103.00,1,103.12686,64,4.93,1.208457,12.12314,1.117556
495919,23,64.83,1,48.37,5,17,NaN,53.42,1,50.44892,79,NaN,NaN,14.38108,1.285062
537705,24,112.27,6,28.95,5,10,10.15,103.75,1,103.12686,64,18.80,2.852217,9.14314,1.088659
567235,39,16.13,0,49.11,1,17,43.17,12.00,2,6.49000,21,5.94,1.137596,9.64000,2.485362
602553,24,112.27,6,28.19,1,10,23.15,103.25,1,103.12686,53,5.04,1.217711,9.14314,1.088659
672139,24,115.74,6,28.95,1,14,23.50,103.25,1,103.12686,35,5.45,1.231915,12.61314,1.122307


In [98]:
df_train[cols_to_compare].drop_duplicates()

,Podcast_Name,Episode_Num,Host_Popularity_percentage,Guest_Popularity_percentage
id,,,,
0,0,98,74.81,NaN
1,1,26,66.95,75.95
2,2,16,69.97,8.97
3,3,45,57.22,78.70
4,4,86,80.07,58.68
...,...,...,...,...
749995,36,25,69.36,NaN
749996,19,21,35.21,NaN
749997,37,51,78.58,84.89


In [106]:
import pandas as pd

df_sub = pd.read_csv(cfg.sub_path, index_col='id')
df_sub['Listening_Time_minutes'] = df_train["Listening_Time_minutes"].median()

df_sub

,Listening_Time_minutes
id,
750000,43.37946
750001,43.37946
750002,43.37946
750003,43.37946
750004,43.37946
...,...
999995,43.37946
999996,43.37946
999997,43.37946


## Leak

In [113]:
import pandas as pd

df_sub = pd.read_csv(cfg.sub_path, index_col='id')
df_sub['Listening_Time_minutes'] = df_train["Listening_Time_minutes"].median()

cols_to_compare = ['Podcast_Name', 'Episode_Num', 'Host_Popularity_percentage', 'Guest_Popularity_percentage']

df_test_with_id = df_test.copy()
df_test_with_id['id'] = df_test_with_id.index
df_test_with_id = df_test_with_id.dropna(subset=['Episode_Length_minutes', 'Guest_Popularity_percentage'])

leaked_rows = df_test_with_id.merge(
    df_train[cols_to_compare + ['Listening_Time_minutes']].drop_duplicates(),
    on=cols_to_compare,
    how='inner'
)
display(leaked_rows)

df_sub.loc[leaked_rows['id'], 'Listening_Time_minutes'] = leaked_rows['Listening_Time_minutes'].values
display(df_sub)
display(df_sub[df_sub['Listening_Time_minutes'] != 43.37946])

,Podcast_Name,Episode_Length_minutes,Genre,Host_Popularity_percentage,Publication_Day,Publication_Time,Guest_Popularity_percentage,Number_of_Ads,Episode_Sentiment,Episode_Num,Host_Guest_Diff,Host_Guest_Ratio,id,Listening_Time_minutes
0,27,66.28,4,50.26,6,14,65.14,2.0,0,58,-14.88,0.771569,750052,65.78956
1,31,105.62,8,72.92,4,21,70.95,0.0,2,28,1.97,1.027766,750384,98.16834
2,36,114.01,2,80.18,0,14,22.72,0.0,1,2,57.46,3.529049,750542,89.17529
3,47,97.78,6,78.78,1,10,0.83,1.0,2,94,77.95,94.915663,750576,64.26326
4,19,42.18,8,76.28,0,14,38.62,1.0,2,27,37.66,1.975142,750627,39.52058
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3866,16,118.59,7,62.00,1,10,83.26,0.0,2,7,-21.26,0.744655,999693,50.72345
3867,3,53.68,3,83.53,3,21,20.06,2.0,1,9,63.47,4.164008,999780,48.21410
3868,3,100.33,3,97.53,0,14,2.76,1.0,2,85,94.77,35.336957,999792,61.03035
3869,47,108.74,6,37.72,6,14,26.91,0.0,2,74,10.81,1.401709,999832,89.81921


,Listening_Time_minutes
id,
750000,43.37946
750001,43.37946
750002,43.37946
750003,43.37946
750004,43.37946
...,...
999995,43.37946
999996,43.37946
999997,43.37946


,Listening_Time_minutes
id,
750052,65.78956
750384,98.16834
750542,89.17529
750576,64.26326
750627,39.52058
...,...
999693,50.72345
999780,48.21410
999792,61.03035


In [14]:
df_dup = df_train.copy()
df_dup = df_dup.dropna(subset=['Episode_Length_minutes', 'Guest_Popularity_percentage'])
df_dup = df_dup[df_dup.duplicated(subset=['Podcast_Name', 'Episode_Num', 'Host_Popularity_percentage', 'Guest_Popularity_percentage'], keep=False)]
# df_dup = df_dup.sort_values(['Podcast_Name', 'Episode_Num', 'Host_Popularity_percentage', 'Guest_Popularity_percentage'])
df_dup = df_dup.sort_values(['Listening_Time_minutes'])

In [17]:
x = 200
for i in range(x, x+5):
    display(df_dup.iloc[i*20:(i+1)*20])

,Podcast_Name,Episode_Length_minutes,Genre,Host_Popularity_percentage,Publication_Day,Publication_Time,Guest_Popularity_percentage,Number_of_Ads,Episode_Sentiment,Listening_Time_minutes,Episode_Num,Host_Guest_Diff,Host_Guest_Ratio,Listening_Episode_Diff,Listening_Episode_Ratio
id,,,,,,,,,,,,,,,
263900,44,44.33,0,26.24,0,14,29.00,1.0,0,36.86970,35,-2.76,0.904828,7.46030,1.202342
536468,30,50.08,1,48.74,4,14,74.67,1.0,1,36.91364,49,-25.93,0.652739,13.16636,1.35668
537609,30,47.82,0,48.74,5,14,74.67,1.0,1,36.91364,49,-25.93,0.652739,10.90636,1.295456
111866,12,56.53,7,96.77,3,17,68.53,1.0,1,36.95000,59,28.24,1.412082,19.58000,1.529905
644291,12,54.95,7,96.77,3,10,68.53,1.0,2,36.95000,59,28.24,1.412082,18.00000,1.487145
43025,1,49.59,1,54.85,0,17,95.29,1.0,1,36.95261,20,-40.44,0.575611,12.63739,1.341989
649585,1,49.59,1,54.85,0,17,95.29,1.0,2,36.95261,20,-40.44,0.575611,12.63739,1.341989
472139,3,44.32,3,91.75,0,21,13.82,0.0,2,36.98696,12,77.93,6.638929,7.33304,1.19826
188514,3,44.89,3,91.75,0,21,13.82,2.0,2,36.98696,12,77.93,6.638929,7.90304,1.213671


,Podcast_Name,Episode_Length_minutes,Genre,Host_Popularity_percentage,Publication_Day,Publication_Time,Guest_Popularity_percentage,Number_of_Ads,Episode_Sentiment,Listening_Time_minutes,Episode_Num,Host_Guest_Diff,Host_Guest_Ratio,Listening_Episode_Diff,Listening_Episode_Ratio
id,,,,,,,,,,,,,,,
612131,47,52.76,7,55.82,6,17,54.53,1.0,0,37.03334,54,1.29,1.023657,15.72666,1.424662
618658,17,81.54,1,27.36,0,10,73.54,1.0,2,37.06000,71,-46.18,0.372042,44.48000,2.200216
90431,18,81.76,7,24.19,0,17,24.19,3.0,1,37.06364,19,0.00,1.0,44.69636,2.205936
281725,18,81.81,7,24.19,5,17,24.19,3.0,0,37.06364,19,0.00,1.0,44.74636,2.207285
474986,42,52.27,8,28.57,5,10,65.91,2.0,0,37.06527,84,-37.34,0.43347,15.20473,1.410215
526658,42,52.27,8,28.57,5,10,65.91,2.0,1,37.06527,84,-37.34,0.43347,15.20473,1.410215
262065,23,62.77,1,40.94,4,17,54.81,1.0,1,37.12852,27,-13.87,0.746944,25.64148,1.690614
145253,23,62.77,1,40.94,2,10,54.81,1.0,1,37.12852,27,-13.87,0.746944,25.64148,1.690614
496728,8,61.52,5,40.75,1,21,70.29,1.0,1,37.14311,14,-29.54,0.579741,24.37689,1.656296


,Podcast_Name,Episode_Length_minutes,Genre,Host_Popularity_percentage,Publication_Day,Publication_Time,Guest_Popularity_percentage,Number_of_Ads,Episode_Sentiment,Listening_Time_minutes,Episode_Num,Host_Guest_Diff,Host_Guest_Ratio,Listening_Episode_Diff,Listening_Episode_Ratio
id,,,,,,,,,,,,,,,
437343,31,39.70,8,82.87,6,10,49.37,0.0,2,37.31164,99,33.50,1.67855,2.38836,1.064011
397046,31,39.97,8,82.87,2,10,49.37,0.0,2,37.31164,99,33.50,1.67855,2.65836,1.071247
648422,16,47.40,7,49.30,3,21,48.40,0.0,2,37.32532,79,0.90,1.018595,10.07468,1.269915
116959,16,49.57,7,49.30,3,21,48.40,0.0,2,37.32532,79,0.90,1.018595,12.24468,1.328053
734928,0,88.76,0,44.15,5,21,16.72,0.0,1,37.35358,49,27.43,2.64055,51.40642,2.376211
558378,0,88.76,0,44.15,5,21,16.72,2.0,1,37.35358,49,27.43,2.64055,51.40642,2.376211
73344,22,45.93,4,20.60,6,17,5.98,1.0,1,37.36460,99,14.62,3.444816,8.56540,1.229238
509299,22,45.55,4,59.60,6,17,5.98,1.0,2,37.36460,99,53.62,9.966555,8.18540,1.219068
428400,22,45.64,4,59.60,6,17,5.98,1.0,1,37.36460,99,53.62,9.966555,8.27540,1.221477


,Podcast_Name,Episode_Length_minutes,Genre,Host_Popularity_percentage,Publication_Day,Publication_Time,Guest_Popularity_percentage,Number_of_Ads,Episode_Sentiment,Listening_Time_minutes,Episode_Num,Host_Guest_Diff,Host_Guest_Ratio,Listening_Episode_Diff,Listening_Episode_Ratio
id,,,,,,,,,,,,,,,
575310,44,59.06,0,78.07,0,17,79.77,0.0,2,37.45874,98,-1.70,0.978689,21.60126,1.576668
369748,27,46.98,0,79.53,0,21,5.66,0.0,1,37.46085,98,73.87,14.051237,9.51915,1.254109
357136,27,46.96,4,79.53,0,21,5.66,0.0,1,37.46085,98,73.87,14.051237,9.49915,1.253575
634361,32,65.62,4,38.01,4,10,74.51,1.0,1,37.48498,62,-36.50,0.510133,28.13502,1.750568
504709,32,65.62,4,38.01,6,10,74.51,0.0,1,37.48498,62,-36.50,0.510133,28.13502,1.750568
573274,20,44.09,3,30.89,5,10,72.25,1.0,1,37.48793,89,-41.36,0.427543,6.60207,1.176112
49536,20,44.09,3,30.89,5,10,72.25,2.0,2,37.48793,89,-41.36,0.427543,6.60207,1.176112
289876,41,37.99,5,70.43,3,10,5.03,1.0,2,37.49978,60,65.40,14.001988,0.49022,1.013073
155822,41,70.46,5,70.43,3,10,5.03,1.0,2,37.49978,60,65.40,14.001988,32.96022,1.878944


,Podcast_Name,Episode_Length_minutes,Genre,Host_Popularity_percentage,Publication_Day,Publication_Time,Guest_Popularity_percentage,Number_of_Ads,Episode_Sentiment,Listening_Time_minutes,Episode_Num,Host_Guest_Diff,Host_Guest_Ratio,Listening_Episode_Diff,Listening_Episode_Ratio
id,,,,,,,,,,,,,,,
136353,33,52.03,9,67.82,3,10,15.73,0.0,1,37.58718,91,52.09,4.311507,14.44282,1.384249
722171,43,65.53,1,95.50,0,10,74.25,2.0,1,37.60622,70,21.25,1.286195,27.92378,1.742531
26312,43,65.52,1,95.50,0,10,74.25,0.0,1,37.60622,70,21.25,1.286195,27.91378,1.742265
388044,47,50.45,6,78.59,1,17,5.36,2.0,1,37.65000,18,73.23,14.662313,12.80000,1.339973
155048,47,50.45,6,78.59,6,14,5.36,2.0,1,37.65000,18,73.23,14.662313,12.80000,1.339973
44091,47,54.68,6,78.59,6,14,5.36,2.0,1,37.65000,18,73.23,14.662313,17.03000,1.452324
426121,45,51.82,8,49.72,3,21,15.33,1.0,1,37.65011,79,34.39,3.243314,14.16989,1.376357
146529,45,52.44,8,49.72,3,14,15.33,1.0,0,37.65011,79,34.39,3.243314,14.78989,1.392825
618568,30,39.69,1,84.56,5,10,89.98,0.0,0,37.66547,54,-5.42,0.939764,2.02453,1.05375


In [93]:
df_dup_diff = df_dup[~df_dup.duplicated(subset=['Podcast_Name', 'Episode_Num', 'Host_Popularity_percentage', 'Listening_Time_minutes'], keep=False)].sort_values(['Podcast_Name', 'Episode_Num', 'Host_Popularity_percentage', 'Guest_Popularity_percentage'])
df_dup_diff

,Podcast_Name,Episode_Length_minutes,Genre,Host_Popularity_percentage,Publication_Day,Publication_Time,Guest_Popularity_percentage,Number_of_Ads,Episode_Sentiment,Listening_Time_minutes,Episode_Num,Host_Guest_Diff,Host_Guest_Ratio,Listening_Episode_Diff,Listening_Episode_Ratio
id,,,,,,,,,,,,,,,
149705,0,33.59,0,58.94,2,10,78.14,2.0,0,12.52697,11,-19.20,0.754287,21.06303,2.681415
208744,0,68.56,0,58.94,4,14,83.21,2.0,0,49.54627,11,-24.27,0.708328,19.01373,1.383757
128674,0,31.31,0,22.17,3,21,86.19,2.0,2,22.12941,15,-64.02,0.257222,9.18059,1.414859
671633,0,28.36,0,22.17,3,21,86.51,2.0,2,19.12941,15,-64.34,0.256271,9.23059,1.482534
212086,0,41.30,0,72.39,3,10,44.86,2.0,0,26.61207,26,27.53,1.613687,14.68793,1.551927
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
276477,47,112.72,6,89.35,2,17,99.69,0.0,2,81.76854,81,-10.34,0.896278,30.95146,1.378525
585266,47,56.42,6,87.79,3,17,84.93,0.0,0,27.78027,86,2.86,1.033675,28.63973,2.030938
727528,47,9.11,6,87.79,4,14,85.65,0.0,0,8.36440,86,2.14,1.024985,0.74560,1.08914


In [ ]:
# Please get only duplicate with Podcast_Name, Episode_Num
df_dup = df_train[df_train.duplicated(subset=['Podcast_Name', 'Episode_Num'], keep=False)]
df_dup = df_dup.sort_values(['Podcast_Name', 'Episode_Num', 'Host_Popularity_percentage', 'Guest_Popularity_percentage'])
df_dup

,Podcast_Name,Episode_Length_minutes,Genre,Host_Popularity_percentage,Publication_Day,Publication_Time,Guest_Popularity_percentage,Number_of_Ads,Episode_Sentiment,Episode_Num
id,,,,,,,,,,
2221,0,55.10,0,68.79,6,14,6.29,1.0,2,1
6639,0,85.75,0,96.60,0,21,9.86,0.0,0,1
12478,0,69.75,0,94.31,4,17,94.08,2.0,0,1
26451,0,63.84,0,95.01,6,10,22.62,0.0,1,1
33972,0,90.33,0,85.02,0,10,27.76,3.0,2,1
...,...,...,...,...,...,...,...,...,...,...
714536,47,50.09,6,67.47,2,10,53.58,0.0,0,100
715096,47,71.71,6,73.30,1,21,53.58,1.0,1,100
717445,47,7.20,6,38.86,0,17,30.73,1.0,1,100


Early stopping, best iteration is:
[9399]	training's rmse: 6.29499	valid_1's rmse: 12.5142
Validation score: 12.51421341907998